# Search Performance Lab

**Optional supporting lab.** This empirical extension is separate from the standard Chapter 23 lab and homework and adds no required meeting.

Measure array linear search, array binary search, List.Contains, and HashSet.Contains on the same generated workload. This restores the earlier empirical lab alongside the current operation-count and complete-workload analysis.

These explanations and tasks retain the earlier CS1 teaching material.[^restored]


## Construct the Experiments

```{index} Construct the Experiments
```

Use positive input size `n`, positive lookup count `rep`, and a recorded random seed. Generate one array of integers. Copy it for sorted binary search, construct a List from it, and construct a HashSet from it. A set removes duplicates, so report both input length and set count.

Generate the query values from the **same original array** for all four experiments. Advance the selected index by `m = Math.Max(1, n / rep)` and wrap around at the end. When `rep > n`, the stride is one and queries cycle through the input. This is a successful-lookup workload; add a separate missing-target workload rather than silently changing the question.

Prepare and sort the data before starting the Stopwatch. Warm up each search first so initial compilation has less influence. Start the stopwatch immediately before the lookup loop, stop immediately afterward, and display the elapsed duration in seconds. Count found results and verify that all four counts equal rep so the program consumes and checks each result.

## Implement and Run

```{index} Implement and Run
```

Create a console project named SearchPerformance and use the complete demonstration below as the first experiment. Run it once unchanged, then parse command-line arguments for `rep` followed by any number of sizes. For example, `dotnet run -- 5000 100 1000 10000`. Reject missing, non-integer, or nonpositive parameters before creating arrays.

1. Print parsed arguments before introducing timing.
2. Run array linear search for one size.
3. Run all four methods for that size and check their found counts.
4. Repeat for all sizes using the same recorded seed.
5. Print a right-aligned table with size, repetitions, set count, and times in seconds.
6. Repeat trials and report variability. Preserve raw durations and units; do not demand nonzero output after rounding.

The demonstration uses a regular stride through the original data, including duplicate targets. Binary search operates on a separately sorted copy, but receives the identical query sequence. No console output or file I/O belongs inside the measured loop.

In [7]:
using System;
using System.Collections.Generic;
using System.Diagnostics;
static bool Linear(int[] values, int target) {
    foreach (int value in values) if (value == target) return true;
    return false;
}
int n = 1000, rep = 2000, seed = 42;
Random random = new Random(seed);
int[] data = new int[n];
for (int i = 0; i < n; i++) data[i] = random.Next();
int[] sorted = (int[])data.Clone(); Array.Sort(sorted);
List<int> list = new List<int>(data);
HashSet<int> set = new HashSet<int>(data);
int stride = Math.Max(1, n / rep);
Func<int, bool>[] searches = {
    target => Linear(data, target),
    target => Array.BinarySearch(sorted, target) >= 0,
    target => list.Contains(target),
    target => set.Contains(target)
};
string[] labels = {"Array linear", "Array binary", "List", "HashSet"};
Console.WriteLine($"n={n}; repetitions={rep}; seed={seed}; set count={set.Count}");
for (int method = 0; method < searches.Length; method++) {
    Func<int, bool> search = searches[method];
    for (int warm = 0; warm < n; warm++) search(data[warm]);
    int found = 0, index = 0;
    Stopwatch watch = Stopwatch.StartNew();
    for (int k = 0; k < rep; k++) {
        if (search(data[index])) found++;
        index = (int)(((long)index + stride) % n);
    }
    watch.Stop();
    if (found != rep) throw new Exception("Search results disagree.");
    Console.WriteLine($"{labels[method],12}: {watch.Elapsed.TotalSeconds:F6} seconds; found={found}");
}

n=1000; repetitions=2000; seed=42; set count=1000
Array linear: 0.001055 seconds; found=2000
Array binary: 0.000123 seconds; found=2000
        List: 0.000102 seconds; found=2000
     HashSet: 0.000056 seconds; found=2000


## Interpret Measurements

```{index} Interpret Measurements
```

The historical lab reduced the linear-search repetitions when `n * rep` grew very large, then scaled its measured duration by `rep / rep2`. If you try this extension, display the **actual** repetition count and label the scaled number as an estimate. It is not a measurement of the full run. Never mix those estimates with direct durations without identifying them.

Exclude data construction only when answering the narrow lookup question. A recommendation for a real system must add sorting/construction, updates, query frequency, and storage costs, as the current workload lesson explains. A HashSet pays for hashing and extra storage; a sorted array pays for ordering; a tiny input can be dominated by fixed costs. Delegate-call overhead in this demonstration also contributes to measured duration, so treat tiny differences cautiously.

Submit your runnable project, exact command, runtime/build mode, sizes, seeds, repetition counts, raw tables, and a short interpretation. Include at least three input sizes and repeated trials. Explain one limitation and how a follow-up experiment would address it. Keep ordinary correctness tests separate from timing assertions: elapsed-time thresholds are not portable correctness tests.

```{rubric} Footnotes
```

[^restored]: Restores the original performance lab: seeded data, regular-stride queries, four search representations, repeated lookups, and Stopwatch. The revised apparatus shares identical query values and distinguishes measured durations from scaled estimates.